# Ordered Logistic Regression Results for Adoption Predictors: FAIR^2 Dataset Exploration with `mlcroissant`
This notebook demonstrates loading, processing, and exploring the FAIR^2 dataset using the [`mlcroissant`](https://pypi.org/project/mlcroissant/) library. All exploration refer to dataset elements by their Croissant `@id`, following reproducible and machine-actionable FAIR practices.

### Dataset Source
This dataset implements the Croissant metadata schema and is accessible via:

`https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json`

In [ ]:
# Install `mlcroissant` (run only if not already installed)
!pip install mlcroissant

## 1. Data Loading
Load dataset metadata and prepare to inspect the available record sets and fields.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the Croissant schema JSON-LD URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Load the dataset metadata using mlcroissant
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata  # This is a mlcroissant.Metadata object

print(f"Dataset Name: {metadata.name}\n\nDescription: {metadata.description}")
print(f"\nPublished: {metadata.date_published}")
print(f"Spatial Coverage: {metadata.spatial_coverage}")
print(f"License: {metadata.license}")
print(f"Croissant Schema Version: {metadata.conforms_to}")

## 2. Data Overview
List the record sets (tables) and fields with their Croissant `@id`s.

- **Record Set `@id`**: uniquely identifies each logical table.
- **Field `@id`**: uniquely identifies each column/field within record sets.

_Note: This step helps you select which data to extract and explore next._

In [ ]:
# List available record sets (tables) in the dataset
record_sets = dataset.record_sets
print("Available record sets and their fields:\n")
for rs in record_sets:
    rs_id = rs.id
    print(f"- RecordSet @id: {rs_id}")
    if hasattr(rs, 'fields'):
        for field in rs.fields:
            print(f"     - Field @id: {field.id}   | name: {field.name}   | type: {field.data_type}")
    if hasattr(rs, 'columns') and rs.columns:
        for col in rs.columns:
            print(f"     - Column @id: {col.id}   | name: {col.name}   | type: {col.data_type}")
    print()

## 3. Data Extraction
Extract the records for each record set into a `pandas` DataFrame. You can then reference specific record set, field, or column `@id`s for focused analysis.

In [ ]:
# Extract all records for all record sets using their @id, store as pandas DataFrames
dataframes = {}
record_set_ids = [rs.id for rs in dataset.record_sets]

for record_set_id in record_set_ids:
    # mlcroissant uses record_set argument as the @id value
    records = list(dataset.records(record_set=record_set_id))
    if records:
        dataframes[record_set_id] = pd.DataFrame.from_records(records)

# Display available dataframes with their columns (fields/columns by @id)
for rsid, df in dataframes.items():
    print(f"RecordSet @id: {rsid}")
    print(f"  Columns: {list(df.columns)}\n")

# Choose a record set for further exploration
if dataframes:
    selected_record_set_id = list(dataframes.keys())[0]  # Use the first one, change as needed
    print(f"\nExample preview of records from `{selected_record_set_id}` record set:")
    display(dataframes[selected_record_set_id].head())
else:
    print("No record sets with data were found.")

## 4. Exploratory Data Analysis (EDA)
Now, perform typical EDA operations such as filtering, normalization, and grouping. **All field selection should use their Croissant field or column `@id`.**

_Replace `numeric_field_id` and `group_field_id` below with actual `@id` values from the overview above. Adjust thresholds and analysis as appropriate for your data._

In [ ]:
# If there is at least one data frame loaded, proceed with EDA
if dataframes:
    record_set_id = selected_record_set_id
    df = dataframes[record_set_id]
    # Identify a numeric field for demo (pick first numeric looking column, change as needed)
    numeric_field_id = None
    for col in df.columns:
        if pd.api.types.is_numeric_dtype(df[col]):
            numeric_field_id = col
            break
    if numeric_field_id is None:
        print("No numeric field found for filtering and normalization; please specify manually.")
    else:
        print(f"Using numeric field: {numeric_field_id}")
        # Filter records: example, value > threshold
        threshold = df[numeric_field_id].mean() if df[numeric_field_id].mean() > 0 else 10
        filtered_df = df[df[numeric_field_id] > threshold].copy()
        print(f"Filtered records where `{numeric_field_id}` > {threshold:.2f}:")
        display(filtered_df.head())

        # Normalize
        filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"Normalized `{numeric_field_id}` for filtered records:")
        display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        # Pick a categorical field for grouping (choose first object type, else skip)
        group_field_id = None
        for col in df.columns:
            if pd.api.types.is_object_dtype(df[col]) and col != numeric_field_id:
                group_field_id = col
                break
        if group_field_id:
            print(f"Grouping by field: {group_field_id}")
            grouped = filtered_df.groupby(group_field_id)[numeric_field_id].mean()
            print(f"Mean `{numeric_field_id}` by `{group_field_id}`:")
            display(grouped.head())
        else:
            print("No categorical (object) field found for grouping; please specify manually.")
else:
    print("No data available for EDA.")

## 5. Visualization
Visualize numeric field distributions and example relationships, referencing Croissant field `@id` in plot labels.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Only visualize if we have a numeric field and filtered dataframe
if 'filtered_df' in locals() and numeric_field_id:
    plt.figure(figsize=(8, 6))
    sns.histplot(filtered_df[numeric_field_id], bins=20, kde=True)
    plt.title(f"Distribution of `{numeric_field_id}` (record set `{record_set_id}`)")
    plt.xlabel(f"{numeric_field_id}")
    plt.ylabel("Count")
    plt.show()

    # If group_field_id available, boxplot by group
    if 'group_field_id' in locals() and group_field_id:
        plt.figure(figsize=(10, 6))
        sns.boxplot(x=filtered_df[group_field_id], y=filtered_df[numeric_field_id])
        plt.title(f"{numeric_field_id} by {group_field_id} (record set: {record_set_id})")
        plt.xlabel(f"{group_field_id}")
        plt.ylabel(f"{numeric_field_id}")
        plt.show()
else:
    print("No numeric field data to visualize.")

## 6. Conclusion
This notebook has demonstrated programmatic exploration of the FAIR^2 dataset using Croissant-compliant access via `mlcroissant`. Key steps included referencing all entities by their `@id`, extracting data into dataframes, and performing EDA and visualizations.

- Ensure that you always refer to record sets, fields, and columns by their Croissant `@id` for reproducibility.
- For more information on dataset structure and provenance, inspect the dataset metadata via `dataset.metadata`.
- For advanced processing or ML workflows, leverage `mlcroissant`'s direct data streaming and semantic mapping capabilities.

Feel free to adapt this template to your own Croissant-based data assets!